# Candidate top 50k coverage + financial wide table

这个 notebook 使用 Companies House Accounts Bulk monthly ZIP，对 `accounts_download_candidates_top.csv` 中的 candidate top 50k 公司进行：

- monthly bulk accounts 覆盖率统计；
- turnover 覆盖率统计；
- candidate company-year 财务宽表抽取；
- `financial_core_score` 与 `financial_conservative_score` 构建；
- evidence tier / metric availability / turnover 兼容输出。

当前 score 命名以 `financial_score_modeling_rules.md` 为准。


In [ ]:
from pathlib import Path
import csv
import html
import json
import math
import os
import re
import zipfile
from collections import Counter, defaultdict
from datetime import datetime

import pandas as pd


def find_repo_root():
    env_root = os.environ.get("PROJECT_ROOT")
    starts = []
    if env_root:
        starts.append(Path(env_root))
    starts.append(Path.cwd())

    for start in starts:
        start = start.resolve()
        for p in [start, *start.parents]:
            if (p / ".git").exists() and (p / "00 Data Preparation + EDA").exists():
                return p
    raise FileNotFoundError(
        "Cannot find repository root. Start Jupyter from inside the cloned repo, "
        "or set PROJECT_ROOT to the repository root."
    )


def require_exists(path, label="path"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path


def first_existing(paths, label):
    paths = [Path(path) for path in paths]
    for path in paths:
        if path.exists():
            return path
    tried = "\n".join(f"- {path}" for path in paths)
    raise FileNotFoundError(f"Cannot find {label}. Tried:\n{tried}")


PROJECT_ROOT = find_repo_root()
FINANCIAL_DIR = PROJECT_ROOT / "00 Data Preparation + EDA" / "Financial Data"
SMALL_TEST_DIR = FINANCIAL_DIR / "Small Scale Data Test"
SAMPLE_COMPANIES_DIR = SMALL_TEST_DIR / "Sample Companies"
LOCAL_DATA_ROOT = Path(os.environ.get(
    "LOCAL_DATA_ROOT",
    r"E:\000硕士毕设\财务数据\Local Large Data",
))
ACCOUNTS_ZIP_DIR = first_existing(
    [
        LOCAL_DATA_ROOT / "Accounts Data_2025.1_2026.5",
        FINANCIAL_DIR / "Accounts Data_2025.1_2026.5",
    ],
    "accounts ZIP directory",
)
BULK_MATCH_DIR = SMALL_TEST_DIR / "2025全年bulk匹配"
MODEL_PREP_DIR = BULK_MATCH_DIR / "模型训练与数据处理"
MODEL_TRAIN_DIR = MODEL_PREP_DIR / "模型训练"

require_exists(FINANCIAL_DIR, "financial data directory")
require_exists(SMALL_TEST_DIR, "small scale test directory")
require_exists(ACCOUNTS_ZIP_DIR, "accounts ZIP directory")


BASE_DIR = BULK_MATCH_DIR
ZIP_DIR = ACCOUNTS_ZIP_DIR
CANDIDATE_FILE = SAMPLE_COMPANIES_DIR / "accounts_download_candidates_top.csv"

# Fallback: if the original 50k candidate file is not available, recover the
# candidate company list from a previously generated matches file.
CANDIDATE_FALLBACK_FILE = BASE_DIR / "candidate_accounts_bulk_matches_all.csv"
if not CANDIDATE_FILE.exists() and CANDIDATE_FALLBACK_FILE.exists():
    CANDIDATE_FILE = CANDIDATE_FALLBACK_FILE

# all_files: 解析 candidate 命中的全部 accounts files，适合 company-year 训练样本。
# latest_per_company: 每家公司只解析最新一份 accounts，适合生成 company-level 最新特征。
MATCH_SELECTION_MODE = "all_files"

# 调试时可设为 500；正式 candidate 测试设为 None。
MAX_FILES = None

BASE_DIR.mkdir(parents=True, exist_ok=True)

# Coverage / match outputs.
OUTPUT_MATCHES_ALL = BASE_DIR / "candidate_accounts_bulk_matches_all.csv"
OUTPUT_MATCHES_LATEST = BASE_DIR / "candidate_accounts_bulk_matches_latest.csv"
OUTPUT_COVERAGE_REPORT = BASE_DIR / "candidate_monthly_coverage_report.json"
OUTPUT_COVERAGE_COMPANY_FLAGS = BASE_DIR / "candidate_monthly_coverage_company_flags.csv"

# Candidate financial proxy outputs.
OUTPUT_FACTS_LONG = BASE_DIR / "financial_facts_long_candidate.csv"
OUTPUT_FEATURES = BASE_DIR / "financial_features_candidate_year.csv"
OUTPUT_AVAILABILITY = BASE_DIR / "financial_metric_availability_candidate_summary.csv"
OUTPUT_EVIDENCE = BASE_DIR / "financial_evidence_tier_candidate_summary.csv"
OUTPUT_ERRORS = BASE_DIR / "financial_extraction_errors_candidate.csv"

# Compatible turnover outputs.
OUTPUT_TURNOVER_FACTS = BASE_DIR / "turnover_facts_long_candidate.csv"
OUTPUT_TURNOVER_SELECTED = BASE_DIR / "turnover_company_year_selected_candidate.csv"
OUTPUT_TURNOVER_SUMMARY = BASE_DIR / "turnover_extraction_summary_candidate.csv"
OUTPUT_TURNOVER_ERRORS = BASE_DIR / "turnover_extraction_errors_candidate.csv"

require_exists(CANDIDATE_FILE, "candidate file")
if not sorted(ZIP_DIR.glob("Accounts_Monthly_Data-*.zip")):
    raise FileNotFoundError(f"No monthly accounts ZIP files found in {ZIP_DIR}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOCAL_DATA_ROOT:", LOCAL_DATA_ROOT)
print("Candidate file:", CANDIDATE_FILE)
print("ZIP dir:", ZIP_DIR)
print("Output dir:", BASE_DIR)
print("Match selection mode:", MATCH_SELECTION_MODE)


## 1. 读取 candidate top 50k 并扫描 monthly ZIP 生成 matches


In [ ]:
FILENAME_RE = re.compile(
    r"Prod\d+(?:_\d+)?_(?P<company>[A-Z0-9]{8})_(?P<period>\d{8})\.(?:html|xhtml|xml)$",
    re.I,
)

def normalise_company_number(value):
    value = "" if pd.isna(value) else str(value).strip().upper()
    if re.fullmatch(r"\d+", value):
        return value.zfill(8)[-8:]
    return value

candidates = pd.read_csv(CANDIDATE_FILE, dtype=str)
candidates["CompanyNumber"] = candidates["CompanyNumber"].map(normalise_company_number)
candidates = candidates.drop_duplicates(subset=["CompanyNumber"]).reset_index(drop=True)
candidate_by_company = candidates.set_index("CompanyNumber").to_dict("index")
target_companies = set(candidate_by_company)

zip_paths = sorted(ZIP_DIR.glob("Accounts_Monthly_Data-*.zip"))
print("Candidate companies:", len(candidates))
print("Monthly ZIP files:", len(zip_paths))

match_rows = []
company_file_count = Counter()
company_zip_count = defaultdict(set)
company_first_period = {}
company_latest_period = {}
zip_summary = []
combined_zip_companies = set()

for zip_path in zip_paths:
    parsed_files = 0
    matched_files = 0
    zip_companies = set()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            name = Path(info.filename).name
            m = FILENAME_RE.search(name)
            if not m:
                continue
            parsed_files += 1
            company = m.group("company").upper()
            period = m.group("period")
            zip_companies.add(company)
            combined_zip_companies.add(company)
            if company not in target_companies:
                continue

            matched_files += 1
            meta = candidate_by_company[company]
            company_file_count[company] += 1
            company_zip_count[company].add(zip_path.name)
            company_first_period[company] = min(company_first_period.get(company, period), period)
            company_latest_period[company] = max(company_latest_period.get(company, period), period)
            match_rows.append({
                **meta,
                "CompanyNumber": company,
                "source_zip": zip_path.name,
                "internal_filename": info.filename,
                "period_end": period,
            })

    zip_summary.append({
        "zip_name": zip_path.name,
        "parsed_account_files": parsed_files,
        "unique_companies": len(zip_companies),
        "matched_candidate_files": matched_files,
    })

matches_all = pd.DataFrame(match_rows)
if matches_all.empty:
    matches_latest = matches_all.copy()
else:
    matches_all = matches_all.drop_duplicates(subset=["source_zip", "internal_filename"]).reset_index(drop=True)
    matches_latest = (
        matches_all.sort_values(["CompanyNumber", "period_end", "source_zip", "internal_filename"])
        .drop_duplicates(subset=["CompanyNumber"], keep="last")
        .reset_index(drop=True)
    )

matches_all.to_csv(OUTPUT_MATCHES_ALL, index=False, encoding="utf-8-sig")
matches_latest.to_csv(OUTPUT_MATCHES_LATEST, index=False, encoding="utf-8-sig")

if MATCH_SELECTION_MODE == "latest_per_company":
    matches = matches_latest.copy()
elif MATCH_SELECTION_MODE == "all_files":
    matches = matches_all.copy()
else:
    raise ValueError("MATCH_SELECTION_MODE must be 'all_files' or 'latest_per_company'")

if MAX_FILES is not None:
    matches = matches.head(MAX_FILES).copy()

account_coverage_flags = []
for company, row in candidate_by_company.items():
    account_coverage_flags.append({
        "CompanyNumber": company,
        "CompanyName": row.get("CompanyName", ""),
        "primary_sector": row.get("primary_sector", ""),
        "Accounts_AccountCategory": row.get("Accounts_AccountCategory", ""),
        "account_category_group": row.get("account_category_group", ""),
        "has_bulk_account": company_file_count[company] > 0,
        "matched_file_count": company_file_count[company],
        "matched_zip_count": len(company_zip_count.get(company, set())),
        "first_period": company_first_period.get(company, ""),
        "latest_period": company_latest_period.get(company, ""),
    })

account_coverage_df = pd.DataFrame(account_coverage_flags)

print("Matched files all:", len(matches_all))
print("Matched companies:", int(account_coverage_df["has_bulk_account"].sum()))
print("Matched company rate:", float(account_coverage_df["has_bulk_account"].mean()))
print("Rows selected for extraction:", len(matches))
display(account_coverage_df.head())
display(matches.head())


## 2. 标签字典与 iXBRL 解析函数

这里使用 exact tag matching 为主，避免把 policy/comment/text disclosure 抽成数值指标。不同 taxonomy 里的 namespace 会被去掉，只保留本地标签名。


In [ ]:
FACT_GROUPS = {
    "turnover": {
        "tags": ["TurnoverRevenue", "Revenue", "SalesRevenue"],
        "kind": "flow",
        "monetary": True,
    },
    "current_assets": {
        "tags": ["CurrentAssets"],
        "kind": "stock",
        "monetary": True,
    },
    "fixed_assets": {
        "tags": ["FixedAssets", "FixedAssetsTotal"],
        "kind": "stock",
        "monetary": True,
    },
    "net_current_assets_liabilities": {
        "tags": ["NetCurrentAssetsLiabilities"],
        "kind": "stock",
        "monetary": True,
    },
    "total_assets_less_current_liabilities": {
        "tags": ["TotalAssetsLessCurrentLiabilities"],
        "kind": "stock",
        "monetary": True,
    },
    "net_assets_liabilities": {
        "tags": [
            "NetAssetsLiabilities",
            "NetAssets",
            "NetAssetsLiabilitiesIncludingPensionAssetLiability",
        ],
        "kind": "stock",
        "monetary": True,
    },
    "equity": {
        "tags": ["Equity", "CapitalAndReserves", "ShareholdersFunds"],
        "kind": "stock",
        "monetary": True,
    },
    "cash": {
        "tags": ["CashBankOnHand", "CashCashEquivalents", "CashAndCashEquivalents"],
        "kind": "stock",
        "monetary": True,
    },
    "debtors": {
        "tags": [
            "Debtors",
            "TradeDebtorsTradeReceivables",
            "OtherDebtors",
        ],
        "kind": "stock",
        "monetary": True,
    },
    # Diagnostic output showed that the useful aggregate in the current sample is
    # "Creditors", while the more specific "amounts falling due..." tags were not observed.
    "creditors_total": {
        "tags": ["Creditors"],
        "kind": "stock",
        "monetary": True,
    },
    "trade_creditors": {
        "tags": ["TradeCreditorsTradePayables"],
        "kind": "stock",
        "monetary": True,
    },
    "other_creditors": {
        "tags": [
            "OtherCreditors",
            "OtherCreditorsIncludingTaxationSocialSecurityBalanceSheetSubtotal",
        ],
        "kind": "stock",
        "monetary": True,
    },
    "accrued_liabilities": {
        "tags": [
            "AccruedLiabilitiesNotExpressedWithinCreditorsSubtotal",
            "AccruedLiabilitiesDeferredIncome",
            "AccruedLiabilities",
        ],
        "kind": "stock",
        "monetary": True,
    },
    "tax_payable": {
        "tags": [
            "OtherTaxationSocialSecurityPayable",
            "TaxationSocialSecurityPayable",
            "CorporationTaxPayable",
            "Value-addedTaxPayable",
        ],
        "kind": "stock",
        "monetary": True,
    },
    "borrowings": {
        "tags": [
            "BankBorrowingsOverdrafts",
            "BankBorrowings",
            "TotalBorrowings",
            "OtherRemainingBorrowings",
            "FinanceLeaseLiabilitiesPresentValueTotal",
            "TotalLiabilities",
            "FinancialLiabilities",
        ],
        "kind": "stock",
        "monetary": True,
    },
    # Keep these specific fields for future taxonomies/files where they appear,
    # but do not rely on them for the current sample.
    "creditors_due_within_one_year": {
        "tags": [
            "CreditorsAmountsFallingDueWithinOneYear",
            "CreditorsDueWithinOneYear",
            "CurrentLiabilities",
            "TotalCurrentLiabilities",
        ],
        "kind": "stock",
        "monetary": True,
    },
    "creditors_due_after_one_year": {
        "tags": [
            "CreditorsAmountsFallingDueAfterMoreThanOneYear",
            "CreditorsDueAfterOneYear",
            "CreditorsAmountsFallingDueAfterOneYear",
            "NoncurrentLiabilities",
        ],
        "kind": "stock",
        "monetary": True,
    },
    "employees": {
        "tags": ["AverageNumberEmployeesDuringPeriod", "AverageNumberEmployees"],
        "kind": "flow",
        "monetary": False,
    },
    "profit_loss": {
        "tags": [
            "ProfitLoss",
            "ProfitLossBeforeTax",
            "OperatingProfitLoss",
            "ProfitLossOnOrdinaryActivitiesBeforeTax",
            "ProfitLossForPeriod",
        ],
        "kind": "flow",
        "monetary": True,
    },
}

TAG_TO_GROUP = {}
TAG_PRIORITY = {}
for group, spec in FACT_GROUPS.items():
    for idx, tag in enumerate(spec["tags"]):
        TAG_TO_GROUP.setdefault(tag, group)
        TAG_PRIORITY[(group, tag)] = len(spec["tags"]) - idx


def strip_namespace(value):
    return str(value).split(":")[-1] if value else ""


def normalise_period_end(value):
    value = "" if pd.isna(value) else str(value)
    digits = re.sub(r"\D", "", value)
    if len(digits) == 8:
        return f"{digits[:4]}-{digits[4:6]}-{digits[6:8]}"
    return ""


def parse_numeric_value(text, scale=None, sign=None):
    if text is None:
        return None
    cleaned = re.sub(r"[\s,\u00a0£$€]", "", str(text))
    if cleaned in {"", "-", "—", "–"}:
        return None
    is_parenthesised_negative = cleaned.startswith("(") and cleaned.endswith(")")
    cleaned = cleaned.strip("()")
    try:
        value = float(cleaned)
    except ValueError:
        return None
    if scale not in (None, ""):
        try:
            value *= 10 ** int(scale)
        except ValueError:
            pass
    if str(sign).strip() == "-" or is_parenthesised_negative:
        value *= -1
    return value


def parse_attrs(text):
    return dict(re.findall(r'([A-Za-z_:][\w:.-]*)\s*=\s*"([^"]*)"', text))


def strip_tags(text):
    return html.unescape(re.sub(r"<[^>]+>", "", text)).strip()


def parse_contexts(text):
    contexts = {}
    context_re = re.compile(r"<[^>]*context\b(?P<attrs>[^>]*)>(?P<body>.*?)</[^>]*context>", re.I | re.S)
    for match in context_re.finditer(text):
        attrs = parse_attrs(match.group("attrs"))
        body = match.group("body")
        context_id = attrs.get("id", "")
        if not context_id:
            continue
        info = {
            "context_ref": context_id,
            "period_type": "",
            "start_date": "",
            "end_date": "",
            "instant_date": "",
            "dimension_count": 0,
        }
        start = re.search(r"<[^>]*startDate[^>]*>(.*?)</[^>]*startDate>", body, re.I | re.S)
        end = re.search(r"<[^>]*endDate[^>]*>(.*?)</[^>]*endDate>", body, re.I | re.S)
        instant = re.search(r"<[^>]*instant[^>]*>(.*?)</[^>]*instant>", body, re.I | re.S)
        if start:
            info["start_date"] = strip_tags(start.group(1))
            info["period_type"] = "duration"
        if end:
            info["end_date"] = strip_tags(end.group(1))
            info["period_type"] = "duration"
        if instant:
            info["instant_date"] = strip_tags(instant.group(1))
            info["period_type"] = "instant"
        info["dimension_count"] = len(re.findall(r"<[^>]*(explicitMember|typedMember)\b", body, re.I))
        contexts[context_id] = info
    return contexts


def parse_ixbrl_document(data):
    text = data.decode("utf-8", "replace") if isinstance(data, bytes) else str(data)
    contexts = parse_contexts(text)
    facts = []
    fact_re = re.compile(r"<[^>]*nonFraction\b(?P<attrs>[^>]*)>(?P<body>.*?)</[^>]*nonFraction>", re.I | re.S)

    for match in fact_re.finditer(text):
        attrs = parse_attrs(match.group("attrs"))
        raw_name = attrs.get("name", "")
        fact_name = strip_namespace(raw_name)
        fact_group = TAG_TO_GROUP.get(fact_name)
        if fact_group is None:
            continue

        raw_value = strip_tags(match.group("body"))
        numeric_value = parse_numeric_value(raw_value, attrs.get("scale"), attrs.get("sign"))
        if numeric_value is None:
            continue

        context_ref = attrs.get("contextRef", "")
        context = contexts.get(context_ref, {})

        facts.append({
            "fact_group": fact_group,
            "fact_name": fact_name,
            "raw_name": raw_name,
            "context_ref": context_ref,
            "unit_ref": attrs.get("unitRef", ""),
            "decimals": attrs.get("decimals", ""),
            "scale": attrs.get("scale", ""),
            "format": attrs.get("format", ""),
            "sign": attrs.get("sign", ""),
            "raw_value": raw_value,
            "numeric_value": numeric_value,
            "context_period_type": context.get("period_type", ""),
            "context_start_date": context.get("start_date", ""),
            "context_end_date": context.get("end_date", ""),
            "context_instant_date": context.get("instant_date", ""),
            "context_dimension_count": context.get("dimension_count", 0),
        })
    return facts


def score_fact(fact, period_end_iso):
    group = fact["fact_group"]
    spec = FACT_GROUPS[group]
    score = 0

    score += TAG_PRIORITY.get((group, fact["fact_name"]), 0) * 5

    if spec["kind"] == "flow":
        if fact.get("context_end_date") == period_end_iso:
            score += 40
        if fact.get("context_period_type") == "duration":
            score += 15
    else:
        if fact.get("context_instant_date") == period_end_iso:
            score += 40
        if fact.get("context_period_type") == "instant":
            score += 15

    if fact.get("context_dimension_count", 0) == 0:
        score += 25
    else:
        score -= 15

    if spec["monetary"]:
        if "GBP" in str(fact.get("unit_ref", "")).upper():
            score += 10
    else:
        score += 10

    return score


## 3. 从 ZIP 流式抽取 facts 并选择每组最佳指标


In [ ]:
facts_long = []
feature_rows = []
error_rows = []

zip_cache = {}

for i, row in matches.iterrows():
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(matches)}")

    zip_name = row["source_zip"]
    internal_filename = row["internal_filename"]
    zip_path = ZIP_DIR / zip_name
    period_end_iso = normalise_period_end(row["period_end"])

    try:
        if zip_name not in zip_cache:
            zip_cache[zip_name] = zipfile.ZipFile(zip_path)

        data = zip_cache[zip_name].read(internal_filename)
        facts = parse_ixbrl_document(data)

        for fact in facts:
            fact["selection_score"] = score_fact(fact, period_end_iso)
            facts_long.append({
                "CompanyNumber": row["CompanyNumber"],
                "CompanyName": row.get("CompanyName", ""),
                "primary_sector": row.get("primary_sector", ""),
                "Accounts_AccountCategory": row.get("Accounts_AccountCategory", ""),
                "account_category_group": row.get("account_category_group", ""),
                "source_zip": zip_name,
                "internal_filename": internal_filename,
                "period_end": row["period_end"],
                "period_end_iso": period_end_iso,
                **fact,
            })

        best_by_group = {}
        for fact in facts:
            group = fact["fact_group"]
            current = best_by_group.get(group)
            if current is None or (
                fact["selection_score"],
                abs(fact["numeric_value"]),
            ) > (
                current["selection_score"],
                abs(current["numeric_value"]),
            ):
                best_by_group[group] = fact

        feature = {
            "CompanyNumber": row["CompanyNumber"],
            "CompanyName": row.get("CompanyName", ""),
            "primary_sector": row.get("primary_sector", ""),
            "Accounts_AccountCategory": row.get("Accounts_AccountCategory", ""),
            "account_category_group": row.get("account_category_group", ""),
            "source_zip": zip_name,
            "internal_filename": internal_filename,
            "period_end": row["period_end"],
            "period_end_iso": period_end_iso,
            "parse_status": "parsed",
            "total_candidate_fact_count": len(facts),
        }

        for group in FACT_GROUPS:
            best = best_by_group.get(group)
            feature[group] = best["numeric_value"] if best else None
            feature[f"{group}_fact_name"] = best["fact_name"] if best else ""
            feature[f"{group}_score"] = best["selection_score"] if best else None
            feature[f"{group}_context_ref"] = best["context_ref"] if best else ""
            feature[f"{group}_context_date"] = (
                best["context_end_date"] if best and FACT_GROUPS[group]["kind"] == "flow"
                else best["context_instant_date"] if best
                else ""
            )

        feature_rows.append(feature)

    except Exception as exc:
        error_rows.append({
            "CompanyNumber": row.get("CompanyNumber", ""),
            "CompanyName": row.get("CompanyName", ""),
            "source_zip": zip_name,
            "internal_filename": internal_filename,
            "period_end": row.get("period_end", ""),
            "error_type": type(exc).__name__,
            "error_message": str(exc)[:500],
        })
        feature_rows.append({
            "CompanyNumber": row.get("CompanyNumber", ""),
            "CompanyName": row.get("CompanyName", ""),
            "primary_sector": row.get("primary_sector", ""),
            "Accounts_AccountCategory": row.get("Accounts_AccountCategory", ""),
            "account_category_group": row.get("account_category_group", ""),
            "source_zip": zip_name,
            "internal_filename": internal_filename,
            "period_end": row.get("period_end", ""),
            "period_end_iso": period_end_iso,
            "parse_status": "error",
            "total_candidate_fact_count": 0,
        })

for z in zip_cache.values():
    z.close()

facts_df = pd.DataFrame(facts_long)
features_df = pd.DataFrame(feature_rows)
errors_df = pd.DataFrame(error_rows)

print("Long facts:", len(facts_df))
print("Feature rows:", len(features_df))
print("Errors:", len(errors_df))
display(features_df.head())
display(facts_df.head())


## 4. Evidence tier、质量分与 proxy score


In [ ]:
PROXY_FIELDS = [
    "current_assets",
    "fixed_assets",
    "net_current_assets_liabilities",
    "total_assets_less_current_liabilities",
    "net_assets_liabilities",
    "equity",
    "cash",
    "debtors",
    "creditors_total",
    "trade_creditors",
    "other_creditors",
    "accrued_liabilities",
    "tax_payable",
    "borrowings",
    "creditors_due_within_one_year",
    "creditors_due_after_one_year",
    "employees",
    "profit_loss",
]

CORE_FINANCIAL_FIELDS = [
    "current_assets",
    "net_assets_liabilities",
    "equity",
    "cash",
    "debtors",
    "creditors_total",
    "employees",
]

CORE_BALANCE_FIELDS = CORE_FINANCIAL_FIELDS

def safe_signed_log1p(value):
    if pd.isna(value):
        return None
    try:
        value = float(value)
        return math.copysign(math.log1p(abs(value)), value)
    except Exception:
        return None

def minmax_0_100(series):
    if series.notna().sum() > 1 and series.max() != series.min():
        return (series - series.min()) / (series.max() - series.min()) * 100
    return pd.Series([None] * len(series), index=series.index)

features_df["has_turnover"] = features_df["turnover"].notna()
features_df["available_proxy_field_count"] = features_df[PROXY_FIELDS].notna().sum(axis=1)
features_df["available_core_balance_field_count"] = features_df[CORE_FINANCIAL_FIELDS].notna().sum(axis=1)

for field in CORE_FINANCIAL_FIELDS:
    features_df[f"has_{field}"] = features_df[field].notna()

features_df["has_balance_sheet_core"] = features_df["available_core_balance_field_count"].ge(4)
features_df["has_employee_count"] = features_df["employees"].notna()

def assign_evidence_tier(row):
    if row["has_turnover"]:
        return "T1_observed_turnover"
    if row["available_core_balance_field_count"] >= 4:
        return "T2_balance_sheet_rich"
    if row["available_proxy_field_count"] >= 2:
        return "T3_balance_sheet_partial"
    return "T4_account_category_only"

features_df["financial_evidence_tier"] = features_df.apply(assign_evidence_tier, axis=1)

log_source_fields = list(dict.fromkeys(["turnover"] + PROXY_FIELDS))
for field in log_source_fields:
    features_df[f"log_{field}"] = features_df[field].map(safe_signed_log1p)

core_log_fields = [f"log_{field}" for field in CORE_FINANCIAL_FIELDS]

features_df["financial_core_score_raw"] = features_df[core_log_fields].mean(axis=1, skipna=True)
features_df["financial_core_score"] = minmax_0_100(features_df["financial_core_score_raw"])

features_df["financial_conservative_score_raw"] = (
    features_df[core_log_fields].fillna(0).sum(axis=1) / len(CORE_FINANCIAL_FIELDS)
)
features_df["financial_conservative_score"] = minmax_0_100(features_df["financial_conservative_score_raw"])

# Backward-compatible aliases for previous sample outputs.
features_df["financial_scale_score_raw"] = features_df["financial_core_score_raw"]
features_df["financial_scale_score"] = features_df["financial_core_score"]

features_df["financial_data_quality_score"] = (
    features_df["has_turnover"].astype(int) * 40
    + features_df["has_balance_sheet_core"].astype(int) * 30
    + features_df["has_employee_count"].astype(int) * 10
    + features_df["available_proxy_field_count"].clip(upper=10) * 2
)

def observed_segment(turnover):
    if pd.isna(turnover):
        return ""
    turnover = float(turnover)
    if turnover < 3_000_000:
        return "BB"
    if turnover < 25_000_000:
        return "SME"
    if turnover < 500_000_000:
        return "MidCorporate"
    return "Large_or_out_of_scope"

features_df["observed_segment_from_turnover"] = features_df["turnover"].map(observed_segment)

display(features_df[[
    "CompanyNumber",
    "CompanyName",
    "Accounts_AccountCategory",
    "turnover",
    "current_assets",
    "net_assets_liabilities",
    "equity",
    "cash",
    "debtors",
    "creditors_total",
    "employees",
    "available_core_balance_field_count",
    "financial_evidence_tier",
    "financial_core_score",
    "financial_conservative_score",
    "financial_data_quality_score",
    "observed_segment_from_turnover",
]].head(20))


## 5. Availability 与 evidence tier 汇总


In [ ]:
metric_fields = list(FACT_GROUPS.keys())

def availability_summary(df, group_cols):
    rows = []
    grouped = df.groupby(group_cols, dropna=False)
    for key, g in grouped:
        if not isinstance(key, tuple):
            key = (key,)
        base = {col: val for col, val in zip(group_cols, key)}
        base["files"] = len(g)
        base["parsed_files"] = int(g["parse_status"].eq("parsed").sum())
        for field in metric_fields:
            base[f"has_{field}_count"] = int(g[field].notna().sum())
            base[f"has_{field}_rate"] = float(g[field].notna().mean())
        rows.append(base)
    return pd.DataFrame(rows).sort_values(group_cols).reset_index(drop=True)

overall_availability = availability_summary(features_df.assign(overall="overall"), ["overall"])
category_availability = availability_summary(features_df, ["Accounts_AccountCategory", "account_category_group"])
sector_availability = availability_summary(features_df, ["primary_sector"])

availability_out = pd.concat(
    [
        overall_availability.assign(summary_level="overall"),
        category_availability.assign(summary_level="account_category"),
        sector_availability.assign(summary_level="sector"),
    ],
    ignore_index=True,
    sort=False,
)

evidence_summary = (
    features_df
    .groupby(["financial_evidence_tier", "Accounts_AccountCategory", "account_category_group"], dropna=False)
    .agg(
        files=("internal_filename", "count"),
        median_core_score=("financial_core_score", "median"),
        median_conservative_score=("financial_conservative_score", "median"),
        median_scale_score=("financial_scale_score", "median"),
        median_quality_score=("financial_data_quality_score", "median"),
        median_core_field_count=("available_core_balance_field_count", "median"),
        observed_turnover_count=("has_turnover", "sum"),
    )
    .reset_index()
    .sort_values(["financial_evidence_tier", "files"], ascending=[True, False])
)

feature_company = (
    features_df.groupby("CompanyNumber", dropna=False)
    .agg(
        parsed_feature_rows=("parse_status", lambda s: int((s == "parsed").sum())),
        extracted_file_rows=("internal_filename", "count"),
        has_turnover=("turnover", lambda s: bool(s.notna().any())),
        turnover_file_count=("turnover", lambda s: int(s.notna().sum())),
        latest_feature_period=("period_end", "max"),
        max_core_score=("financial_core_score", "max"),
        max_conservative_score=("financial_conservative_score", "max"),
        strongest_evidence_tier=("financial_evidence_tier", "min"),
    )
    .reset_index()
)

coverage_company_flags = account_coverage_df.merge(feature_company, on="CompanyNumber", how="left")
coverage_company_flags["has_turnover"] = coverage_company_flags["has_turnover"].fillna(False)
coverage_company_flags["turnover_file_count"] = coverage_company_flags["turnover_file_count"].fillna(0).astype(int)

def coverage_by(df, group_cols):
    return (
        df.groupby(group_cols, dropna=False)
        .agg(
            companies=("CompanyNumber", "count"),
            matched_companies=("has_bulk_account", "sum"),
            turnover_companies=("has_turnover", "sum"),
            matched_files=("matched_file_count", "sum"),
            turnover_files=("turnover_file_count", "sum"),
        )
        .reset_index()
        .assign(
            matched_rate=lambda x: x["matched_companies"] / x["companies"],
            turnover_rate_all=lambda x: x["turnover_companies"] / x["companies"],
            turnover_rate_matched=lambda x: x["turnover_companies"] / x["matched_companies"].replace(0, pd.NA),
        )
    )

coverage_overall = coverage_by(coverage_company_flags.assign(overall="overall"), ["overall"])
coverage_by_category = coverage_by(coverage_company_flags, ["Accounts_AccountCategory", "account_category_group"])
coverage_by_sector = coverage_by(coverage_company_flags, ["primary_sector"])

coverage_summary_out = pd.concat(
    [
        coverage_overall.assign(summary_level="overall"),
        coverage_by_category.assign(summary_level="account_category"),
        coverage_by_sector.assign(summary_level="sector"),
    ],
    ignore_index=True,
    sort=False,
)

coverage_report = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "candidate_file": str(CANDIDATE_FILE),
    "monthly_zip_dir": str(ZIP_DIR),
    "output_dir": str(BASE_DIR),
    "match_selection_mode": MATCH_SELECTION_MODE,
    "zip_count": len(zip_paths),
    "zip_names": [p.name for p in zip_paths],
    "combined_unique_companies_in_zips": len(combined_zip_companies),
    "candidate_companies": int(len(candidates)),
    "matched_companies": int(coverage_company_flags["has_bulk_account"].sum()),
    "matched_rate": float(coverage_company_flags["has_bulk_account"].mean()),
    "matched_files_all": int(len(matches_all)),
    "extracted_files": int(len(features_df)),
    "turnover_companies": int(coverage_company_flags["has_turnover"].sum()),
    "turnover_rate_all_candidates": float(coverage_company_flags["has_turnover"].mean()),
    "turnover_rate_matched": float(
        coverage_company_flags["has_turnover"].sum() / coverage_company_flags["has_bulk_account"].sum()
        if coverage_company_flags["has_bulk_account"].sum() else 0
    ),
    "turnover_files": int(coverage_company_flags["turnover_file_count"].sum()),
    "parse_errors": int(len(errors_df)),
    "evidence_tier_counts_file_level": features_df["financial_evidence_tier"].value_counts(dropna=False).to_dict(),
    "zip_summary": zip_summary,
}

display(overall_availability)
display(category_availability)
display(evidence_summary)
display(coverage_overall)
display(coverage_by_category)


## 6. 兼容 turnover 输出


In [ ]:
turnover_facts_df = facts_df[facts_df["fact_group"].eq("turnover")].copy()

turnover_selected_df = features_df[[
    "CompanyNumber",
    "CompanyName",
    "primary_sector",
    "Accounts_AccountCategory",
    "account_category_group",
    "source_zip",
    "internal_filename",
    "period_end",
    "period_end_iso",
    "parse_status",
    "total_candidate_fact_count",
    "turnover",
    "turnover_fact_name",
    "turnover_score",
    "turnover_context_ref",
    "turnover_context_date",
]].copy()
turnover_selected_df["has_turnover_candidate"] = turnover_selected_df["turnover"].notna()
turnover_selected_df = turnover_selected_df.rename(columns={
    "total_candidate_fact_count": "total_fact_count",
    "turnover": "selected_turnover_value",
    "turnover_fact_name": "selected_fact_name",
    "turnover_score": "selection_score",
    "turnover_context_ref": "selected_context_ref",
    "turnover_context_date": "selected_context_end_date",
})
turnover_selected_df["selected_fact_type"] = turnover_selected_df["selected_fact_name"].where(
    turnover_selected_df["selected_fact_name"].eq(""),
    "turnover_total",
)
turnover_selected_df["turnover_candidate_fact_count"] = (
    turnover_facts_df.groupby(["source_zip", "internal_filename"]).size()
    .reindex(
        pd.MultiIndex.from_frame(turnover_selected_df[["source_zip", "internal_filename"]]),
        fill_value=0,
    )
    .to_numpy()
)

def turnover_summary(df, group_cols):
    return (
        df.groupby(group_cols, dropna=False)
        .agg(
            files=("internal_filename", "count"),
            parsed_files=("parse_status", lambda s: (s == "parsed").sum()),
            files_with_turnover=("has_turnover_candidate", "sum"),
            median_selected_turnover=("selected_turnover_value", "median"),
        )
        .reset_index()
        .assign(
            parse_success_rate=lambda x: x["parsed_files"] / x["files"],
            turnover_candidate_rate=lambda x: x["files_with_turnover"] / x["files"],
        )
    )

turnover_summary_out = pd.concat(
    [
        turnover_summary(turnover_selected_df.assign(group="overall"), ["group"]).assign(summary_level="overall"),
        turnover_summary(turnover_selected_df, ["Accounts_AccountCategory", "account_category_group"]).assign(summary_level="account_category"),
        turnover_summary(turnover_selected_df, ["primary_sector"]).assign(summary_level="sector"),
    ],
    ignore_index=True,
    sort=False,
)

display(turnover_summary_out.head(20))


## 7. 写出结果文件


In [ ]:
facts_df.to_csv(OUTPUT_FACTS_LONG, index=False, encoding="utf-8-sig")
features_df.to_csv(OUTPUT_FEATURES, index=False, encoding="utf-8-sig")
availability_out.to_csv(OUTPUT_AVAILABILITY, index=False, encoding="utf-8-sig")
evidence_summary.to_csv(OUTPUT_EVIDENCE, index=False, encoding="utf-8-sig")
errors_df.to_csv(OUTPUT_ERRORS, index=False, encoding="utf-8-sig")

turnover_facts_df.to_csv(OUTPUT_TURNOVER_FACTS, index=False, encoding="utf-8-sig")
turnover_selected_df.to_csv(OUTPUT_TURNOVER_SELECTED, index=False, encoding="utf-8-sig")
turnover_summary_out.to_csv(OUTPUT_TURNOVER_SUMMARY, index=False, encoding="utf-8-sig")
errors_df.to_csv(OUTPUT_TURNOVER_ERRORS, index=False, encoding="utf-8-sig")

coverage_company_flags.to_csv(OUTPUT_COVERAGE_COMPANY_FLAGS, index=False, encoding="utf-8-sig")
OUTPUT_COVERAGE_REPORT.write_text(json.dumps(coverage_report, ensure_ascii=False, indent=2), encoding="utf-8")
coverage_summary_out.to_csv(BASE_DIR / "candidate_monthly_coverage_summary.csv", index=False, encoding="utf-8-sig")

print("Wrote:")
for path in [
    OUTPUT_MATCHES_ALL,
    OUTPUT_MATCHES_LATEST,
    OUTPUT_COVERAGE_REPORT,
    OUTPUT_COVERAGE_COMPANY_FLAGS,
    BASE_DIR / "candidate_monthly_coverage_summary.csv",
    OUTPUT_FACTS_LONG,
    OUTPUT_FEATURES,
    OUTPUT_AVAILABILITY,
    OUTPUT_EVIDENCE,
    OUTPUT_ERRORS,
    OUTPUT_TURNOVER_FACTS,
    OUTPUT_TURNOVER_SELECTED,
    OUTPUT_TURNOVER_SUMMARY,
    OUTPUT_TURNOVER_ERRORS,
]:
    print(path)


## 8. 输出检查建议

优先检查：

- `candidate_monthly_coverage_report.json`：candidate top 50k 的 accounts / turnover 覆盖率。
- `candidate_monthly_coverage_summary.csv`：按 account category / sector 的覆盖率差异。
- `financial_features_candidate_year.csv`：candidate company-year 财务宽表。
- `financial_metric_availability_candidate_summary.csv`：各财务指标覆盖率。
- `financial_evidence_tier_candidate_summary.csv`：T1/T2/T3/T4 分布。
- `turnover_company_year_selected_candidate.csv`：T1 observed turnover 标签。

如果要改成每家公司只保留最新 accounts，把第 1 个代码单元中的：

```text
MATCH_SELECTION_MODE = "all_files"
```

改成：

```text
MATCH_SELECTION_MODE = "latest_per_company"
```
